In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.5'
}

In [3]:
drama_name = []
aired = []
ratings = []
no_of_ratings = []
synopsis = []
screenwriter = []
director = []
genre = []
country = []
Type = []
episodes = []
original_network = []
duration = []
ranking = []
popularity = []
content_rating = []

In [4]:
def extract_people_and_genres(ul):
    sc = "N/A"
    dr = "N/A"
    genres = "N/A"

    if not ul:
        return sc, dr, genres

    for li in ul.find_all("li", class_="list-item"):
        b = li.find("b")
        if not b:
            continue

        label = b.text.replace(":", "").strip()

        # Screenwriter
        if label == "Screenwriter":
            sc = ", ".join(a.text.strip() for a in li.find_all("a"))

        # Director
        elif label == "Director":
            dr = ", ".join(a.text.strip() for a in li.find_all("a"))

        # Screenwriter & Director (combined case)
        elif label == "Screenwriter & Director":
            names = ", ".join(a.text.strip() for a in li.find_all("a"))
            sc = dr = names

        # Genres
        elif label == "Genres":
            genres = ", ".join(a.text.strip() for a in li.find_all("a"))

    return sc, dr, genres

In [5]:
def extract_key_value_list(ul):
    data = {}
    if not ul:
        return data

    for li in ul.find_all("li", class_="list-item"):
        b = li.find("b")
        if not b:
            continue

        key = b.text.replace(":", "").strip()
        value = li.get_text(" ", strip=True).replace(b.text, "").strip()
        data[key] = value

    return data

In [6]:
def extract_stats(soup):
    rating = "N/A"
    rating_count = "N/A"

    rating_box = soup.find("div", class_="hfs", itempropx="aggregateRating")
    if rating_box:
        rating = rating_box.find("b").text.strip()
        rating_count = rating_box.text.split("from")[-1].replace("users", "").strip()

    return rating, rating_count

In [ ]:
for j in range(1,11):
    url = f"https://mydramalist.com/shows/popular?page={j}"

    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content,"html.parser")
        element = soup.find_all("h6",class_ = "text-primary title")
        for i in range(0,20):
            try:
                link = "https://mydramalist.com" + element[i].find("a").get("href")
                new_response = requests.get(link, headers=headers)
                new_soup = BeautifulSoup(new_response.content,"html.parser")
                
                d_title = new_soup.find("h1",class_ ="film-title")
                drama_name.append(d_title.text if d_title else "N/A")
                
                rating, rating_count = extract_stats(new_soup)
                ratings.append(rating)
                no_of_ratings.append(rating_count)

                sn = new_soup.find("div", class_="show-synopsis")
                synopsis.append(
                    sn.get_text(" ", strip=True).replace("Edit Translation", "") if sn else "N/A"
                )
                
                list_of_items_1 = new_soup.find("ul", class_="list m-a-0")

                sc, dr, gn = extract_people_and_genres(list_of_items_1)
                
                screenwriter.append(sc)
                director.append(dr)
                genre.append(gn)
                
                list_of_items_2 = new_soup.find("ul", class_="list m-a-0 hidden-md-up")
                info = extract_key_value_list(list_of_items_2)
                
                country.append(info.get("Country", "N/A"))
                Type.append(info.get("Type", "N/A"))
                episodes.append(info.get("Episodes", "N/A"))
                aired.append(info.get("Aired", "N/A"))
                original_network.append(info.get("Original Network", "N/A"))
                duration.append(info.get("Duration", "N/A"))
                ranking.append(info.get("Ranked", "N/A"))
                popularity.append(info.get("Popularity", "N/A"))
                content_rating.append(info.get("Content Rating", "N/A"))
                
            except Exception as e:
                continue
    except Exception as e:
        continue

In [ ]:
df = pd.DataFrame({
    "drama name" : drama_name ,"aired" : aired ,"ratings" : ratings,"no of ratings" : no_of_ratings, "synopsis" : synopsis,
    "screenwriter" : screenwriter, "director" : director , "genre" : genre ,"country" : country , "episodes" : episodes , "type":Type,
    "original network" : original_network ,"duration" : duration ,"ranking" : ranking , "popularity" : popularity ,"content rating" : content_rating
})

In [ ]:
df

In [ ]:
pd.to_csv(df,"drama.csv")